# MyBabyAI Cloud Trainer (Optimized)

This notebook provides a professional training environment for the **CodeMind** model with advanced monitoring, multi-dataset support (English/Turkish), and VRAM optimization.

In [ ]:
# 1. Install dependencies
!pip install torch torchvision transformers accelerate peft bitsandbytes sentencepiece tokenizers safetensors huggingface-hub chromadb sentence-transformers PyYAML python-dotenv rich tqdm numpy pandas pillow datasets requests beautifulsoup4 psutil

In [ ]:
# 2. Clone Repository & Force Update
import os
import sys

REPO_URL = "https://github.com/halilibrahimavsar/mybabyai.git"
REPO_DIR = "mybabyai"

if not os.path.exists(REPO_DIR):
    print("Klonlanıyor...")
    !git clone {REPO_URL}
    os.chdir(REPO_DIR)
else:
    print("Güncelleniyor (Force Pull)...")
    %cd {REPO_DIR}
    # Discard any local changes and pull latest from main/master
    !git fetch --all
    !git reset --hard origin/main || !git reset --hard origin/master
    !git pull

sys.path.insert(0, os.getcwd())
print(f"Çalışma dizini: {os.getcwd()}")

### 3. Initialize Model and Trainer
We use the **CodeMind** architecture. If no checkpoint is found (e.g., fresh clone), the system will automatically fall back to a fresh model initialization.

In [ ]:
from src.core.model_manager import ModelManager
from src.core.trainer import LoRATrainer
from src.utils.config import Config

config = Config()
model_manager = ModelManager(config)

# Load CodeMind with fresh model fallback for cloud environments
model_manager.load_model("codemind", allow_fresh_fallback=True)

trainer = LoRATrainer(model_manager, config)

### 4. Advanced Training with Monitoring
You can now train using a pool of datasets (English + Turkish + Custom) and watch the progress in a live-updating rich dashboard.

In [ ]:
# Define your dataset pool
dataset_pool = [
    {
        "name": "English Multi-Turn (UltraChat)",
        "type": "huggingface",
        "dataset_key": "ultrachat_200k",
        "max_samples": 2000
    },
    {
        "name": "Turkish Instructions (Merve)",
        "type": "huggingface",
        "dataset_key": "turkish_instructions_merve",
        "max_samples": 1500
    },
    {
        "name": "Logic & Reasoning (Dolly)",
        "type": "huggingface",
        "dataset_key": "databricks-dolly-15k",
        "max_samples": 1000
    },
    {
        "name": "Coding Basics (Python)",
        "type": "huggingface",
        "dataset_key": "flytech/python-codes-25k",
        "max_samples": 500
    }
]

metrics = trainer.train_from_pool(
    dataset_pool,
    use_notebook_callback=True, # Live Rich Dashboard
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4
)

print("Training Complete!", metrics)

In [ ]:
# 5. Save the final model
!zip -r fine_tuned_codemind.zip codemind/checkpoints models/fine_tuned